# 01 Graph Representation and Timeline Operations

Author: Endri Sojeva

This notebook uses a fictional archival example set in 1840–1920. It builds a causal graph and tries edits on independent branches. Numeric years establish chronological order. Edges describe causal direction, which can differ from that order.

## Setup

Select the project's Python environment. The Kaggle edition loads the same Python package from a source snapshot tied to a GitHub commit.

In [ ]:
import platform
import subprocess
from importlib.metadata import version
from io import BytesIO

import matplotlib.pyplot as plt
import timetravel
from IPython.display import Image, display
from timetravel.graph import build_graph, get_event
from timetravel.models import CausalRelation, Event, RelationKind, StateChange
from timetravel.operations import (
    add_event, add_relation, cancel_travel, modify_event, modify_relation,
    prevent_travel, remove_event, remove_relation, reschedule_event, travel_to_past,
)
from timetravel.visualization import draw_timeline

source_commit = globals().get("SOURCE_COMMIT")
if source_commit is None:
    try:
        source_commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        source_commit = "local checkout (commit unavailable)"
print({"source_commit": source_commit, "python": platform.python_version(),
       "timetravel": timetravel.__version__, "networkx": version("networkx"),
       "matplotlib": version("matplotlib")})

## Construct a causal graph

The receipt asserts that the chart has revision A. The archive receipt provides a causal dependency for a researcher's later departure.

In [ ]:
receipt = Event("receipt", "Archive receives a chart", 1840, "Archive",
                ("chart",), (StateChange("chart", "revision", "A"),),
                importance=2, modification_cost=4)
departure = Event("departure", "Researcher returns the chart", 1920,
                  entities=("chart",), modification_cost=3)
original = build_graph([receipt, departure], [CausalRelation("receipt", "departure")])
print("Base graph:", list(original.edges))
print("Recorded consequence:", get_event(original, "receipt").state_changes)

## Create and edit one branch

Add a catalog entry, then change its description and year. Create a causal relation, change its edit cost, and demonstrate removal of the relation and then the event.

In [ ]:
added = add_event(original, Event("catalog", "Catalog entry", 1900, modification_cost=2))
updated = modify_event(added.graph, Event("catalog", "Index prepared", 1900, modification_cost=2))
moved = reschedule_event(updated.graph, "catalog", 1905)
linked = add_relation(moved.graph, CausalRelation("receipt", "catalog", 2))
priced = modify_relation(linked.graph, CausalRelation("receipt", "catalog", 3))
unlinked = remove_relation(priced.graph, "receipt", "catalog")
removed = remove_event(unlinked.graph, "catalog")
assert "catalog" not in original and "catalog" not in removed.graph
assert get_event(priced.graph, "catalog").time == 1905
print("Branch edit history:", [(edit.operation, edit.cost)
                               for edit in removed.graph.graph["timeline_history"]])

## Travel, cancel and prevent

A journey from 1920 to the existing receipt in 1840 adds a backward edge, completing a two-edge causal loop. Its classification records travel direction. Logical validity requires separate rules.

In [ ]:
trip = travel_to_past(original, "chart-trip", "departure", "receipt", cost=2)
assert trip.graph.edges["departure", "receipt"]["kind"] == RelationKind.TIME_TRAVEL
assert not original.has_edge("departure", "receipt")
print("Travel edges:", [(u, v, data["kind"].value) for u, v, data in trip.graph.edges(data=True)])
figure = draw_timeline(trip.graph)
with BytesIO() as image_buffer:
    figure.savefig(image_buffer, format="png", bbox_inches="tight")
    display(Image(data=image_buffer.getvalue()))
plt.close(figure)

cancelled = cancel_travel(trip.graph, "chart-trip")
prevented = prevent_travel(original, "chart-trip")
assert trip.graph.has_edge("departure", "receipt")
assert not cancelled.graph.has_edge("departure", "receipt")
print("Action states:", trip.record.after.status.value, cancelled.record.after.status.value,
      prevented.record.after.status.value)

## Reject invalid edits

Try a trip to a later event, an already prevented trip and a reschedule that makes a normal causal edge backward. Each rejected edit should leave its input graph usable.

In [ ]:
def expect_rejection(label, operation):
    try:
        operation()
    except ValueError as error:
        print(f"{label}: {error}")
    else:
        raise AssertionError(f"Unexpectedly accepted: {label}")

expect_rejection("Forward trip", lambda: travel_to_past(original, "forward", "receipt", "departure"))
expect_rejection("Prevented trip", lambda: travel_to_past(prevented.graph, "chart-trip", "departure", "receipt"))
expect_rejection("Invalid reschedule", lambda: reschedule_event(original, "departure", 1830))
assert get_event(original, "departure").time == 1920 and list(original.edges) == [("receipt", "departure")]

## Interpretation

The output reports the real edit history, directed dependencies, action statuses and rejected operations. The graph before the trip remains available as an independent starting point. This demonstration exercises the temporal model. Logical paradox detection and performance measurements are reserved for later notebooks.